# 27｜孟加拉国DIA小规模可行性审计与Pilot清单冻结

本笔记本不进行全量孟加拉国分析，也不把宽窗DIA原始MS/MS直接送入按照DDA/标准谱图开发的PBT模型。它首先审计正、负模式64个本地`mzXML`文件，冻结一个带重复、现场空白和QC的小规模试验集合，并确认四窗口DIA采集结构与后续软件条件。

Pilot包含Buriganga城市点B1、上游Meghna农村点M1、下游汇流点M4的双重复，以及FieldBlank1和QC1；正负模式合计16次进样。原始文件只读，不复制、不移动、不改名。

In [ ]:
from pathlib import Path
import json, os, subprocess, time
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = Path(r'E:\谱图预测PBT_终版')
ENV_MS = Path(r'D:\Appinstallation\Anaconda\envs\env_ms\python.exe')
SRC = ROOT / 'src'
CASE = ROOT / 'external_data' / 'bangladesh_surface_water'
OUT = CASE / '01_raw' / '27_pilot_gate'
MANIFEST_DIR = CASE / 'manifests'

MSCONVERT_EXE = Path(r'C:\Users\Lenovo\AppData\Local\Apps\ProteoWizard 3.0.26260.f8f0a9a6 64-bit\msconvert.exe')

# 尚未安装MS-DIAL时保持None即可；27号仍会正常完成数据审计。
# 安装后可改为例如 Path(r'D:\MSDIAL\MsdialConsoleApp.exe') 再运行。
MSDIAL_EXE = None
RUN_AUDIT = True

assert ROOT.is_dir(), ROOT
assert ENV_MS.is_file(), ENV_MS
assert SRC.is_dir(), SRC
assert CASE.is_dir(), CASE
env = os.environ.copy()
env['PYTHONPATH'] = str(SRC) + os.pathsep + env.get('PYTHONPATH', '')
env['PYTHONIOENCODING'] = 'utf-8'

print('env_ms:', ENV_MS)
print('Bangladesh case:', CASE)
print('output:', OUT)

## 1. 科学设计与不可越过的边界

原研究采集了Dhaka周边四条河流12个地点的双重复样品：B为Buriganga、M为Meghna、S为Shitalakshya、T为Turag。B、S、T属于城市相关地点；M1代表较上游农村Meghna；M4位于来自Dhaka河流汇入后的下游Meghna，因此小试选取B1–M1–M4形成城市—农村上游—下游汇流对照。

仪器为Q Exactive HF‑X，每个MS1循环跟随4个DIA MS2扫描，隔离中心为m/z 210、450、690和930，窗口宽度245。每张原始DIA谱包含大量共隔离前体碎片，必须经过MS‑DIAL等工具解卷积形成伪MS/MS后，才能评价是否适合冻结模型。

原论文：[Environmental Science & Technology, DOI 10.1021/acs.est.2c08200](https://pubs.acs.org/doi/10.1021/acs.est.2c08200)；MS‑DIAL官方页面：[CompMS MS-DIAL](https://systemsomicslab.github.io/compms/msdial/main.html)。

## 2. 执行文件清单、哈希与DIA扫描审计

该单元读取全部64个文件名和下载校验记录，并完整顺序读取16个Pilot文件以同时核对SHA256、扫描总数、MS1/MS2比例、峰数和窗口设置。不会解码或修改原始峰数组。通常需要几十秒至数分钟，取决于磁盘速度；主要使用CPU和磁盘，不需要GPU。

In [ ]:
if RUN_AUDIT:
    cmd = [str(ENV_MS), '-m', 'pbt_msms.bangladesh_dia_pilot_gate',
           '--root', str(ROOT), '--msconvert-exe', str(MSCONVERT_EXE)]
    if MSDIAL_EXE is not None:
        cmd += ['--msdial-exe', str(MSDIAL_EXE)]
    print('Running:', ' '.join(cmd))
    started = time.perf_counter()
    subprocess.run(cmd, check=True, cwd=ROOT, env=env)
    print(f'Finished in {(time.perf_counter() - started) / 60:.2f} min')

gate_path = MANIFEST_DIR / '27_feasibility_gate_manifest.json'
qc_path = MANIFEST_DIR / '27_feasibility_qc_checks.csv'
assert gate_path.is_file(), gate_path
gate = json.loads(gate_path.read_text(encoding='utf-8'))
qc = pd.read_csv(qc_path)

allowed_status = {
    'PILOT_DATA_READY_MSDIAL_INSTALL_REQUIRED',
    'READY_FOR_NOTEBOOK_28_DIA_DECONVOLUTION',
}
assert gate['status'] in allowed_status, gate
assert gate['data_qc_all_required_pass'] is True
assert qc.loc[qc['required'].astype(bool), 'passed'].astype(bool).all()

display(pd.Series(gate['counts'], name='count').to_frame())
display(qc)

## 3. 冻结的Pilot进样清单

共8个样品单元、16次进样：6个环境样品（3个地点×双重复）、1个现场空白和1个混合QC，各自包含正负模式。`mzml_output_path`只规定28号的输出位置，本笔记本不执行格式转换。

In [ ]:
pilot = pd.read_csv(OUT / '27_pilot_run_manifest.csv')
conversion = pd.read_csv(OUT / '27_pilot_conversion_plan.csv')
assert len(pilot) == 16
assert pilot['sample_unit_id'].nunique() == 8

display(pilot[[
    'pilot_order', 'sample_unit_id', 'mode', 'sample_type', 'site_id',
    'river', 'setting', 'pilot_contrast', 'filename', 'size_bytes'
]].reset_index(drop=True))

print('Pilot raw size (GB):', round(pilot['size_bytes'].sum() / 1024**3, 2))
print('Planned mzML directory:', CASE / '02_mzml' / '27_pilot')

## 4. DIA采集与原始谱峰数结果

这里的“≥6个峰”只说明原始DIA扫描具有足够的信号内容，**不代表可以直接输入模型**。宽245 m/z的谱仍是混合谱；真正的模型可用率要等28号解卷积后，根据每个MS1特征对应的伪MS/MS计算。

In [ ]:
audit = pd.read_csv(OUT / '27_pilot_mzxml_scan_audit.csv')
display(audit[[
    'sample_unit_id', 'mode', 'sample_type', 'declared_scan_n', 'parsed_scan_n',
    'ms1_scan_n', 'ms2_scan_n', 'ms2_per_ms1_ratio', 'median_ms2_peak_n',
    'ms2_ge6_peak_fraction', 'dia_window_centers_mz', 'dia_window_widths_mz',
    'collision_energies', 'sha256_matches_download_manifest'
]].reset_index(drop=True))

display(Image(filename=str(OUT / 'figures' / '27_pilot_design_and_dia_acquisition.png')))

## 5. 是否进入28号

如果最终状态是：

- `PILOT_DATA_READY_MSDIAL_INSTALL_REQUIRED`：数据和Pilot均已通过，只缺MS‑DIAL。安装官方Windows版本后，把顶部`MSDIAL_EXE`改成实际程序路径并重新运行27号；
- `READY_FOR_NOTEBOOK_28_DIA_DECONVOLUTION`：数据和软件均已就绪，可以制作并运行28号；
- `FAILED_QC`：不要继续，先排查失败项。

28号将先用现有ProteoWizard把冻结的16个`mzXML`标准化为`mzML`，随后进行MS‑DIAL四窗口DIA解卷积，并用空白、QC和双重复评价伪MS/MS质量。未经解卷积的原始DIA谱禁止进入PBT模型。

In [ ]:
print('Notebook 27 final status:', gate['status'])
print('Required data QC all pass:', gate['data_qc_all_required_pass'])
print('MSConvert:', gate['msconvert_executable'])
print('MS-DIAL:', gate['msdial_executable'])
print('Free disk (GB):', round(gate['free_disk_bytes'] / 1024**3, 1))
print('Estimated minimum needed for Pilot conversion/deconvolution (GB):',
      round(gate['estimated_minimum_free_bytes_for_conversion_and_deconvolution'] / 1024**3, 1))
print('Next step:', gate['next_step'])

if not gate['msdial_ready']:
    display(Markdown('**数据门控已经通过；当前只需安装MS-DIAL，暂时不要直接运行原始DIA谱模型推理。**'))
else:
    display(Markdown('**Pilot数据与MS-DIAL均已就绪，可以进入28号DIA解卷积。**'))